# DiTFlow on Wan2.1 — Colab

Runtime → Change runtime type → **GPU** (T4 works for the 1.3B; A100 for the 14B).

Colab's disk is temporary: `/content` is wiped when the runtime recycles.
Download or copy to Drive before you lose results.

In [1]:
!git clone https://github.com/jnheinrich451-eng/ditflow.git
%cd ditflow

fatal: destination path 'ditflow' already exists and is not an empty directory.
/content/ditflow


In [21]:
# Already cloned in an earlier session? Just update.
%cd /content/ditflow
!git pull

/content/ditflow
remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 1.31 KiB | 1.31 MiB/s, done.
From https://github.com/jnheinrich451-eng/ditflow
   5f071ad..f47e975  main       -> origin/main
Updating 5f071ad..f47e975
Fast-forward
 colab_utils.py | 29 +++++++++++++++++++++++++++--
 1 file changed, 27 insertions(+), 2 deletions(-)


## Install

No conda, and **do not reinstall torch** — Colab's build is already correct and
replacing it usually breaks the runtime. Only the missing packages are added.

`transformers>=5` breaks diffusers' model imports, so that upper pin matters.

In [3]:
!pip install -q "diffusers>=0.33" "transformers>=4.44,<5" accelerate ftfy                omegaconf einops imageio imageio-ffmpeg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 140.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.24.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


**Restart the session now** (Runtime → Restart session) — `transformers` almost
certainly changed version. Then continue from the next cell.

In [1]:
%cd /content/ditflow
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content/ditflow
name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


## Verify before spending GPU time

Weight-free, CPU, seconds. Checks the port's AMF against this repo's own
`guidance_utils/motion_flow_utils.py` at f64, plus the transformer chain.
If this fails, nothing downstream is trustworthy.

In [2]:
!python verify_wan_port.py

torch 2.11.0+cu128

=== AMF equivalence vs guidance_utils/motion_flow_utils.py ===
  [PASS] amf argmax=False temp=2: max|diff| = 1.776e-15
  [PASS] amf argmax=False temp=5: max|diff| = 2.665e-15
  [PASS] amf argmax=True  temp=2: max|diff| = 0.000e+00
  [PASS] amf argmax=True  temp=5: max|diff| = 0.000e+00
  [PASS] amf checkpoint_pairs invariant: dq=0.000e+00 dk=0.000e+00

=== transformer / guidance chain ===
  [PASS] full forward: (1, 4, 3, 8, 10)
  [PASS] rotary rewrite identical to diffusers: max|diff| = 0.000e+00
  [PASS] Q/K layout (B,S,heads,D), no text prefix: (1, 60, 2, 8), S=60==60
  [PASS] early exit: (1, 60, 16) token-space
  [PASS] feature hook (t,d,h,w): (3, 16, 4, 5)
  [PASS] gradient reaches latent: |grad| = 1.9687e+00
  [PASS] gradient reaches rope: |grad| = 3.2978e+00
  [PASS] gradient checkpointing invariant: max|diff| = 0.000e+00
  [PASS] KV injection effective: changes output by 1.804e-01

=== config extraction (from_pretrained path) ===
  [PASS] __init__ exposes the

## Generate

First run downloads Wan2.1-T2V-1.3B from HuggingFace (~14 GB, mostly the
umT5-XXL text encoder). Public weights — no token needed.

The bundled clips are 24 frames, so `num_frames` must be a 4k+1 value ≤ 24;
the config defaults to 21. Add `--low_vram` on a <24 GB GPU, `--verbose` to
watch the guidance loss fall.

In [19]:
# most targeted — kill the geometric outliers
!python motion_guidance_wan.py -v ./assets/bmx-trees.mp4 -p "..." --flow_max_disp 10

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers
Fetching 2 files: 100% 2/2 [00:00<00:00, 25343.23it/s]
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.75it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:  20% 1/5 [00:00<00:02,  1.97it/s]
Loading checkpoint shards:  40% 2/5 [00:01<00:01,  1.88it/s]
Loading checkpoint shards:  60% 3/5 [00:01<00:01,  1.95it/s]
Loading checkpoint shards:  80% 4/5 [00:02<00:00,  1.96it/s]
Loading checkpoint shards: 100% 5/5 [00:02<00:00,  2.14it/s]
Loading pipeline components...: 100% 5/5 [00:03<00:00,  1.66it/s]
video model loaded
latent 6x60x104 -> 6 x 30x52 patches, seq_

In [ ]:
# gate on ambiguity directly, via attention peak confidence
!python motion_guidance_wan.py -v ./assets/bmx-trees.mp4 -p "..." --flow_min_conf 0.02

In [ ]:
# stop any survivor from dominating
!python motion_guidance_wan.py -v ./assets/bmx-trees.mp4 -p "..." --flow_loss huber

In [29]:
!python motion_guidance_wan.py     --video_path ./assets/bmx-trees.mp4     --prompt "Leopard running up a snowy hill in a forest"     --verbose

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers
Fetching 2 files: 100% 2/2 [00:00<00:00, 23497.50it/s]
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.64it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:  20% 1/5 [00:00<00:01,  2.12it/s]
Loading checkpoint shards:  40% 2/5 [00:00<00:01,  2.10it/s]
Loading checkpoint shards:  60% 3/5 [00:01<00:00,  2.05it/s]
Loading checkpoint shards:  80% 4/5 [00:01<00:00,  1.97it/s]
Loading checkpoint shards: 100% 5/5 [00:02<00:00,  2.14it/s]
Loading pipeline components...: 100% 5/5 [00:03<00:00,  1.63it/s]
video model loaded
latent 6x60x104 -> 6 x 30x52 patches, seq_

In [25]:
# 1. the outlier filter, fresh output dir
!python motion_guidance_wan.py -v ./assets/lucia.mp4 -p "Cat walks in a city lane" \
    --flow_max_disp 10 --verbose --output_path ./results_lucia_clip




Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers
Fetching 2 files: 100% 2/2 [00:00<00:00, 30283.78it/s]
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.11it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:  20% 1/5 [00:00<00:01,  2.13it/s]
Loading checkpoint shards:  40% 2/5 [00:00<00:01,  2.09it/s]
Loading checkpoint shards:  60% 3/5 [00:01<00:00,  2.06it/s]
Loading checkpoint shards:  80% 4/5 [00:01<00:00,  2.04it/s]
Loading checkpoint shards: 100% 5/5 [00:02<00:00,  2.24it/s]
Loading pipeline components...: 100% 5/5 [00:02<00:00,  1.73it/s]
video model loaded
latent 6x60x104 -> 6 x 30x52 patches, seq_

In [27]:
# 2. if still weak, gate on ambiguity directly
!python motion_guidance_wan.py -v ./assets/lucia.mp4 -p "Cat walks in a city lane" \
    --flow_max_disp 10 --flow_min_conf 0.02 --verbose --output_path ./results_lucia_conf

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Loading Wan-AI/Wan2.1-T2V-1.3B-Diffusers
Fetching 2 files: 100% 2/2 [00:00<00:00, 31655.12it/s]
Loading checkpoint shards: 100% 2/2 [00:00<00:00,  3.49it/s]
Loading pipeline components...:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:   0% 0/5 [00:00<?, ?it/s]
Loading checkpoint shards:  20% 1/5 [00:00<00:01,  2.15it/s]
Loading checkpoint shards:  40% 2/5 [00:00<00:01,  2.11it/s]
Loading checkpoint shards:  60% 3/5 [00:01<00:00,  2.08it/s]
Loading checkpoint shards:  80% 4/5 [00:01<00:00,  2.06it/s]
Loading checkpoint shards: 100% 5/5 [00:02<00:00,  2.25it/s]
Loading pipeline components...: 100% 5/5 [00:02<00:00,  1.73it/s]
video model loaded
latent 6x60x104 -> 6 x 30x52 patches, seq_

## Did guidance actually run?

A suspiciously fast run usually means guidance was skipped. This reports whether
optimised tensors were saved, and echoes the config that produced the output.

In [16]:
from colab_utils import summarize_run
summarize_run("results_wan")

results_wan/
  reference : original.mp4
  generated : Leopard_running_up_a_snowy_hill_in_a_forest.mp4  (0.4 MB)
  generated : Bear_running_in_the_snow_forest.mp4  (0.2 MB)
  generated : Car_walks_in_a_city_lane.mp4  (0.3 MB)
  generated : Cat_walks_in_a_city_lane.mp4  (0.4 MB)
  guidance  : 10 optimised tensor(s) saved, kind(s)=['latent']
              -> guidance DID run on those timesteps
  config    : guidance_blocks=[15], motion_temp=2, num_frames=21, guidance_timestep_range=[50, 40], optimization_steps=5, loss_type=flow, guidance_mode=latent


## Watch it

In [30]:
from colab_utils import show_run
show_run("results_wan")          # reference beside result

In [26]:
from colab_utils import show_run
show_run("results_lucia_clip")          # reference beside result

In [28]:
from colab_utils import show_run
show_run("results_lucia_conf")          # reference beside result

## Get the files off Colab

`zip_results` triggers a browser download and skips `embeds/*.pt` (large, only
needed for `--inject_embeds`). `save_to_drive` survives runtime recycling.

In [6]:
from colab_utils import zip_results
zip_results("results_wan")

/content/ditflow/results_wan.zip  (1.0 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

'/content/ditflow/results_wan.zip'

In [ ]:
# Alternative: keep results across runtime restarts.
# from colab_utils import save_to_drive
# save_to_drive("results_wan")

## Sweep

The defaults are transplanted from CogVideoX and **not tuned for Wan**.
`motion_temp` is the highest-leverage knob: Wan applies RMSNorm to q/k, so
attention logits sit on a different scale than the temperature was tuned for.

`--include_baselines` adds backbone and injection-only rows — without them you
can't tell whether guidance is buying motion fidelity or the backbone just looks
good. Resumable, and writes `results.md` after every run.

In [ ]:
!pip install -q opencv-python
!pip install -q git+https://github.com/openai/CLIP.git      # CLIP score
# motion fidelity pulls CoTracker via torch.hub automatically

In [ ]:
!python sweep_wan.py     -v ./assets/bmx-trees.mp4     -p "Leopard running up a snowy hill in a forest"     --motion_temp 1 2 4     --include_baselines     --output_root ./sweeps/first --dry_run     # drop --dry_run to actually run

In [ ]:
from IPython.display import Markdown
Markdown(open("sweeps/first/results.md").read())

In [2]:
# 1. cut MiraData  (stride 2 from 30 fps -> 24 frames, 1.6 s)
!python benchmark/cut.py --batch "E:\MiraData 9K\81 frames" --dst E:\bench\packed\miradata --split miradata

# 2. join RealCam-Vid to the CUT clips
!python benchmark/realcam.py --clips E:\bench\packed\miradata --realcam E:\RealCam-Vid --dst E:\bench\packed\miradata_traj --manifest benchmark\miradata.csv --split miradata --prompt-classes caption,subject

# 3. cut DAVIS
!python benchmark/cut.py --batch E:\DAVIS\JPEGImages\480p --dst E:\bench\packed\davis --src-fps 24 --split davis --manifest benchmark\davis.csv

# 4. plan, then run
!python benchmark/sweep.py --manifest benchmark\miradata.csv --out E:\bench\runs --dry-run


000000000001.5.002.mp4 -> E:\bench\packed\miradata\000000000001.5.002  stride=2 eff_fps=15.0 1.6s
000000000035.1.003.mp4 -> E:\bench\packed\miradata\000000000035.1.003  stride=2 eff_fps=15.0 1.6s
000000000064.3.002.mp4 -> E:\bench\packed\miradata\000000000064.3.002  stride=2 eff_fps=15.0 1.6s
000000000100.0.003.mp4 -> E:\bench\packed\miradata\000000000100.0.003  stride=2 eff_fps=15.0 1.6s
000000000119.5.007.mp4 -> E:\bench\packed\miradata\000000000119.5.007  stride=2 eff_fps=15.0 1.6s
000000000130.0.015.mp4 -> E:\bench\packed\miradata\000000000130.0.015  stride=2 eff_fps=15.0 1.6s
000000000191.6.037.mp4 -> E:\bench\packed\miradata\000000000191.6.037  stride=2 eff_fps=15.0 1.6s
000000000191.6.038.mp4 -> E:\bench\packed\miradata\000000000191.6.038  stride=2 eff_fps=15.0 1.6s
000000000200.1.006.mp4 -> E:\bench\packed\miradata\000000000200.1.006  stride=2 eff_fps=15.0 1.6s
000000000201.1.008.mp4 -> E:\bench\packed\miradata\000000000201.1.008  stride=2 eff_fps=15.0 1.6s
000000000217.0.006.m

manifest benchmark\miradata.csv line 3: prompt still marked TODO (000000000001.5.002/subject) -- edit it or drop the row
